# 02. Supervised Modeling & Evaluation — Telco Customer Churn
**Author:** Farid Masood Khan  
**Objective:** Train, cross-validate, and evaluate 3 distinct algorithms (Logistic Regression, Random Forest, Gradient Boosting) using leak-free preprocessing.  
**Curriculum Alignment:** DeepLearning.AI / Stanford Online Machine Learning Specialization (Course 1-2: Classification, Regularization, Ensembles, Trees).

---
### Pipeline Workflow:
1. Leak-free stratified train/test split (80/20).
2. Fit `ColumnTransformer` (StandardScaler + OneHotEncoder) strictly on `X_train`.
3. 5-Fold Stratified Cross-Validation on `X_train`.
4. Holdout evaluation on `X_test` (N=1,409) with honest metrics and error analysis.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('../src').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

from preprocess import prepare_splits
from train import get_candidate_models, perform_cross_validation, fit_and_save_models


## 1. Leak-Free Preprocessing & Stratified Split

In [ ]:
data_path = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
X_train_trans, X_test_trans, y_train, y_test, preprocessor, feature_names = prepare_splits(
    data_path=data_path,
    test_size=0.20,
    random_state=42
)

print(f"X_train shape: {X_train_trans.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test_trans.shape}, y_test shape:  {y_test.shape}")
print(f"Train Churn Rate: {y_train.mean():.4f} | Test Churn Rate: {y_test.mean():.4f}")


## 2. 5-Fold Stratified Cross-Validation (Training Split)

In [ ]:
models = get_candidate_models(random_state=42)
cv_results = perform_cross_validation(models, X_train_trans, y_train, cv_folds=5)

cv_summary_df = pd.DataFrame([
    {
        "Model": name,
        "CV ROC-AUC": f"{scores['cv_roc_auc_mean']:.4f} +/- {scores['cv_roc_auc_std']:.4f}",
        "CV F1": f"{scores['cv_f1_mean']:.4f}",
        "CV Recall": f"{scores['cv_recall_mean']:.4f}",
        "CV Accuracy": f"{scores['cv_accuracy_mean']:.4f}",
    }
    for name, scores in cv_results.items()
])
cv_summary_df


## 3. Fit Models on Full Train Split & Evaluate on Holdout Test Split

In [ ]:
test_eval = {}
for name, model in models.items():
    model.fit(X_train_trans, y_train)
    preds = model.predict(X_test_trans)
    probas = model.predict_proba(X_test_trans)[:, 1]
    
    test_eval[name] = {
        "Accuracy": (y_test == preds).mean(),
        "ROC-AUC": roc_auc_score(y_test, probas),
        "Classification Report": classification_report(y_test, preds, target_names=['Retained', 'Churned']),
        "Confusion Matrix": confusion_matrix(y_test, preds)
    }

for name, res in test_eval.items():
    print("=" * 60)
    print(f"MODEL: {name} | Test Accuracy: {res['Accuracy']:.4f} | Test ROC-AUC: {res['ROC-AUC']:.4f}")
    print("=" * 60)
    print(res["Classification Report"])


## 4. Comparative Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=120)
for i, (name, res) in enumerate(test_eval.items()):
    cm = res["Confusion Matrix"]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False, annot_kws={'size': 12, 'weight': 'bold'})
    axes[i].set_title(f"{name}", fontweight='bold')
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")
    axes[i].set_xticklabels(['Stay', 'Churn'])
    axes[i].set_yticklabels(['Stay', 'Churn'])
plt.tight_layout()
plt.show()


## 5. Honest Error Analysis
Which churners does the best model fail to predict?
- Long-tenure subscribers (average tenure of False Negatives is ~33.5 months, compared to ~12.1 months for True Positives).
- Churners with 1-year or 2-year contracts whose cancellation was triggered by exogenous life events rather than steady dissatisfaction.
